# detmatmul — Cross-Hardware Verification

**Deterministic matrix multiplication with SHA-256 verification.**

This notebook runs the canonical 31-case test suite and produces a `hash_manifest.json`.
The SHA-256 hashes it generates will be **identical** to those produced on any other
compliant hardware — NVIDIA, AMD, CPU, any OS.

### What to do
1. Run all cells (`Runtime → Run all`)
2. Download the `hash_manifest.json` that appears at the end
3. Open an issue at [github.com/StanAntonov/detmatmul](https://github.com/StanAntonov/detmatmul)
   titled `Verification: [your hardware]` and attach the file

GPU runtime recommended but not required — CPU-only works fine.

In [ ]:
# Install detmatmul
!pip install git+https://github.com/StanAntonov/detmatmul.git -q

In [ ]:
# Detect hardware
import platform

gpu_name = 'CPU-only'
gpu_sm   = 'N/A'
try:
    from numba import cuda
    dev      = cuda.get_current_device()
    gpu_name = dev.name.decode() if isinstance(dev.name, bytes) else dev.name
    cc       = dev.compute_capability
    gpu_sm   = f'{cc[0]}.{cc[1]}'
    print(f'GPU : {gpu_name}  (sm_{gpu_sm.replace(".","")})')
except Exception:
    print('No GPU detected — running CPU-only mode')

print(f'OS  : {platform.system()} {platform.release()}')
print(f'CPU : {platform.processor() or "unknown"}')

In [ ]:
# Quick sanity check — deterministic matmul
import numpy as np
from detmatmul import matmul, spec_hash, verify_hash

rng = np.random.default_rng(42)
A   = rng.standard_normal((256, 256)).astype(np.float32)
B   = rng.standard_normal((256, 256)).astype(np.float32)

C = matmul(A, B)
h = spec_hash(A, B)

print(f'Output shape : {C.shape}')
print(f'SHA-256      : {h}')
print()
print('Expected     : 0a957271d84451bed064a258a4ee9e933f8ed05752319d7dc4f55f0a4c53fe50')
print()
match = h == '0a957271d84451bed064a258a4ee9e933f8ed05752319d7dc4f55f0a4c53fe50'
print('✅ MATCH — this hardware is compliant' if match else '❌ MISMATCH — investigate')

In [ ]:
# Run the full 31-case canonical test suite
from detmatmul.manifest import build_manifest, save_manifest

force_cpu = (gpu_name == 'CPU-only')

print('Running 31 canonical test cases...')
print('(This takes ~1 min on CPU, ~10s on GPU)\n')

manifest = build_manifest(
    force_cpu = force_cpu,
    gpu_name  = gpu_name,
    gpu_sm    = gpu_sm,
)

hashes = manifest['runs'][0]['hashes']
print(f'Completed {len(hashes)}/31 cases\n')
print(f'{"Test case":<45}  SHA-256 (first 32 chars)')
print('-' * 70)
for k, h in list(hashes.items())[:8]:
    print(f'{k:<45}  {h[:32]}...')
print(f'... and {len(hashes)-8} more')

In [ ]:
# Compare against the reference manifest from the repo
import urllib.request, json
from detmatmul.manifest import merge_manifests, compare_manifests

url = 'https://raw.githubusercontent.com/StanAntonov/detmatmul/main/manifests/hash_manifest.json'
try:
    with urllib.request.urlopen(url) as r:
        reference = json.loads(r.read())
    print(f'Reference manifest loaded ({len(reference["runs"])} platform(s))')

    merged = merge_manifests([reference, manifest])
    result = compare_manifests(merged)

    agreed = result['agreed']
    total  = result['total']
    print(f'\nResult: {agreed}/{total} cases agree')

    if agreed == total:
        print()
        print('╔══════════════════════════════════════════════════════════╗')
        print('║  PROOF COMPLETE                                         ║')
        print(f'║  {gpu_name[:50]:<50}  ║')
        print('║  All 31 hashes match the reference.                    ║')
        print('║  This hardware is DIS v1.0 compliant.                  ║')
        print('╚══════════════════════════════════════════════════════════╝')
    else:
        print(f'\n❌ {result["failed"]} mismatch(es) — please open an issue')
except Exception as e:
    print(f'Could not load reference manifest: {e}')
    print('Saving local manifest only — attach to a GitHub issue manually')

In [ ]:
# Save and download the manifest
save_manifest(manifest, 'hash_manifest.json')
print('Saved hash_manifest.json')

try:
    from google.colab import files
    files.download('hash_manifest.json')
    print('Downloading...')
except ImportError:
    print('Not running in Colab — find hash_manifest.json in your working directory')

print()
print('Next step:')
print('  Open an issue at github.com/StanAntonov/detmatmul')
print(f'  Title: "Verification: {gpu_name}"')
print('  Attach: hash_manifest.json')